# Generate reports with fine-tuned MedGemma LoRA

base MedGemma model + your LoRA adapter


In [1]:
import os
import json
import math
import unicodedata

import numpy as np
import pandas as pd
import torch

from tqdm.auto import tqdm
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel
# os.environ["CUDA_VISIBLE_DEVICES"] = "1"

os.environ["WANDB_DISABLED"] = "true"
os.environ["WANDB_MODE"] = "disabled"
os.environ["WANDB_SILENT"] = "true"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"



In [2]:
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM GB:", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 1))

CUDA available: True
GPU: NVIDIA RTX PRO 6000 Blackwell Server Edition
VRAM GB: 95.0


## Configuration

In [3]:
BASE_MODEL_DIR = "../../FineTuneGeneration/HuggingFace_Models/google/medgemma-27b-text-it"

LORA_ADAPTER_DIR = "../../FineTuneGeneration/models/Ddkd_1_2_medgemma-27b-report-generation-lora/"

TEST_FILE = "data/test_split_sampled.csv"


INPUT_FILE = "data/test_split_sampled.csv"         
OUTPUT_FOLDER = "../Ouptut/Experiments/output_ddkd1_2"    

BATCH_SIZE = 1                         
GREEDY = True                          
SYSTEM_PROMPT = False                  
MERGE_LORA = False                     

MAX_INPUT_LENGTH = 8192
MAX_NEW_TOKENS_GREEDY = 1200
MAX_NEW_TOKENS_SAMPLING = 1300

os.makedirs(OUTPUT_FOLDER, exist_ok=True)

## Prompt formatting functions

Notebook version for `format_prompt.py`

In [4]:
_rng = np.random.default_rng(42)


def normalize_text(value):
    if value is None:
        return ""
    value = str(value).strip().lower()
    value = unicodedata.normalize("NFKD", value).encode("ascii", "ignore").decode("utf-8")
    return value


def convert_cerb_to_her2_score_random(value, p_zero=0.7, rng=None):
    v = normalize_text(value)
    rng = rng or _rng

    if v == "positif":
        return "3+"
    if v == "douteux":
        return "2+"
    if v == "negatif":
        return rng.choice(["0", "1+"], p=[p_zero, 1 - p_zero])
    if v in {"unknown", ""}:
        return "unknown"
    return "unknown"


def _safe_int(value):
    try:
        return int(str(value).strip())
    except (TypeError, ValueError):
        return None


def create_donnees_cliniques(entry_data):
    in_situ = {
        "carcinome in situ",
        "carcinome intracanalaire non infiltrant",
        "carcinome lobulaire in situ",
        "carcinome canalaire in situ",
        "adenocarcinome papillaire intracanalaire non infiltrant",
        "adenocarcinome in situ",
        "carcinome intracanalaire et carcinome lobulaire in situ",
    }

    donnees_cliniques = ""

    # 1) type diagnostique
    type_diag = entry_data.get("type_diagnostique", "unknown")
    if type_diag != "unknown":
        donnees_cliniques += f"- Échantillon : {type_diag}\n"
    else:
        type_diag = "tumorectomie"
        if (
            entry_data.get("taille_tumor_0", "unknown") == "unknown"
            or (
                entry_data.get("ganglions_preleves", "unknown") == "unknown"
                and entry_data.get("taille_tumor_0", "unknown") == "0"
            )
        ):
            type_diag = "biopsie"
        donnees_cliniques += f"- Échantillon : {type_diag}\n"

    # 2) morphologie
    morpho_raw = entry_data.get("ref_morpho_name_tumor_0", "unknown")
    if morpho_raw != "unknown":
        morpho = (
            "carcinome canalaire in situ"
            if morpho_raw == "carcinome intracanalaire non infiltrant"
            else morpho_raw
        )
        donnees_cliniques += f"- Diagnostic (morphologie) : {morpho}\n"

    # 3) grade SBR
    if morpho_raw in in_situ:
        donnees_cliniques += "- Grade SBR : non applicable (lésion non infiltrante)\n"
    elif entry_data.get("ref_grade_tumor_0", "unknown") == "unknown":
        donnees_cliniques += "- Grade SBR : non évalué\n"
    else:
        donnees_cliniques += f"- Grade SBR : {entry_data.get('ref_grade_tumor_0', '')}\n"

    # 4) taille tumorale
    taille = entry_data.get("taille_tumor_0", "unknown")
    if taille != "unknown" and type_diag != "biopsie":
        donnees_cliniques += f"- Taille tumorale : {taille} mm\n"
    elif taille == "unknown":
        if type_diag in {"biopsie", "curage ganglionnaire"}:
            donnees_cliniques += f"- Taille tumorale : non applicable ({type_diag})\n"
        else:
            donnees_cliniques += "- Taille tumorale : non évaluée\n"

    # 5) ganglions examinés
    ganglions_preleves = entry_data.get("ganglions_preleves", "unknown")
    if ganglions_preleves != "unknown":
        donnees_cliniques += f"- Nombre de ganglions examinés : {ganglions_preleves}\n"
    else:
        donnees_cliniques += "- Nombre de ganglions examinés : non évalués\n"

    # 6) ganglions atteints
    ganglions_atteints = entry_data.get("ganglions_atteints", "unknown")
    if ganglions_preleves != "unknown" and ganglions_atteints != "unknown":
        donnees_cliniques += (
            f"- Nombre de ganglions atteints : {ganglions_atteints} "
            f"({ganglions_atteints}/{ganglions_preleves})\n"
        )
    elif ganglions_preleves != "unknown" and ganglions_atteints == "unknown":
        donnees_cliniques += "- Nombre de ganglions atteints : non spécifié\n"

    # 7) emboles vasculaires
    emb = entry_data.get("embols_vasculaires_tumor_0", "unknown")
    if emb != "unknown":
        if emb == "1":
            donnees_cliniques += "- Présence d’emboles vasculaires\n"
        else:
            donnees_cliniques += "- Absence d’emboles vasculaires\n"
    else:
        donnees_cliniques += "- Emboles vasculaires : non évalués\n"

    # 8) rupture capsulaire
    rupture_caps = str(entry_data.get("rupture_capsulaire", "unknown")).strip().lower()
    gang_atteints_int = _safe_int(ganglions_atteints)
    has_positive_nodes = gang_atteints_int is not None and gang_atteints_int > 0

    if rupture_caps == "non":
        donnees_cliniques += "- Présence de rupture capsulaire : non\n"
    elif rupture_caps == "oui" and has_positive_nodes:
        donnees_cliniques += "- Présence de rupture capsulaire : oui\n"
    else:
        donnees_cliniques += "- Présence de rupture capsulaire : non évaluée\n"

    # 9) récepteurs hormonaux
    re = entry_data.get("re_tumor_0", "unknown")
    if re == "unknown":
        donnees_cliniques += "- RE ou RO (récepteurs aux œstrogènes) : non évalués\n"
    elif re == "positifs":
        donnees_cliniques += (
            f"- RE ou RO (récepteurs aux œstrogènes) : positifs ; "
            f"pourcentage : {entry_data['re_perc']}%\n"
        )
    else:
        donnees_cliniques += "- RE ou RO (récepteurs aux œstrogènes) : négatifs\n"

    rp = entry_data.get("rp_tumor_0", "unknown")
    if rp == "unknown":
        donnees_cliniques += "- RP (récepteurs à la progestérone) : non évalués\n"
    elif rp == "positifs":
        donnees_cliniques += (
            f"- RP (récepteurs à la progestérone) : positifs ; "
            f"pourcentage : {entry_data['rp_perc']}%\n"
        )
    else:
        donnees_cliniques += "- RP (récepteurs à la progestérone) : négatifs\n"

    # 10) HER2 / CerbB2
    cerb2 = convert_cerb_to_her2_score_random(
        entry_data.get("cerb_tumor_0", "unknown"),
        p_zero=0.7,
    )
    if cerb2 != "unknown":
        donnees_cliniques += f"- HER2 (CerbB2) : {cerb2}\n"
    else:
        donnees_cliniques += "- HER2 (CerbB2) : non évalué\n"

    # 11) Ki67
    ki67 = entry_data.get("ki67_tumor_0", "unknown")
    if ki67 != "unknown":
        donnees_cliniques += f"- Prolifération cellulaire (Ki67) : {ki67}%\n"
    else:
        donnees_cliniques += "- Prolifération cellulaire (Ki67) : non évaluée\n"

    # 12) marges
    marges = entry_data.get("marges_saines_tumor_0", "unknown")
    if marges != "unknown":
        donnees_cliniques += f"- Marges chirurgicales saines : {marges}\n"
    else:
        if type_diag == "biopsie":
            donnees_cliniques += "- Marges chirurgicales saines : non applicable (biopsie)\n"
        else:
            donnees_cliniques += "- Marges chirurgicales saines : non évaluées\n"

    return donnees_cliniques

In [5]:
def generate_report_prompt(donnees_cliniques, system_prompt=False):
    user_prompt = f"""Tache:
Rédige un **compte rendu anatomopathologique complet en français médical** à partir des données cliniques délimitées ### DONNÉES CLINIQUES ### ci-dessous.

### DONNÉES CLINIQUES ###

{donnees_cliniques}

### FIN DONNÉES CLINIQUES ###


Tu dois produire exactement les sections suivantes :
- Titre (indiquer une latéralité et un quadrant choisis de façon plausible)
- Examen macroscopique
- Examen microscopique
- Étude immunohistochimique
- Conclusion

────────────────────────────────────────
RÈGLES OBLIGATOIRES
────────────────────────────────────────
1) Cohérence générale
- Toute valeur fixe fournie doit être reprise exactement, sans modification.
- Ne jamais faire varier une même donnée entre les sections.

2) Données non évaluées
- Toute donnée indiquée comme "non évaluée", "non réalisée" ou "non disponible" dans les DONNÉES CLINIQUES doit être reprise strictement telle quelle.
- Ne jamais la compléter, estimer ou interpréter.
- Ne jamais la remplacer par une valeur normale ou pathologique (ex : "marges saines", "absence d’emboles", etc.).

3) Compléments descriptifs autorisés
- Tu peux enrichir la description macroscopique et microscopique avec des éléments plausibles, purement descriptifs et rédactionnels.
- Tu ne dois jamais modifier, compléter, interpréter ni contredire les données cliniques fournies.
- Les variables cliniques (taille tumorale, ganglions, marges, emboles, récepteurs, etc.) doivent être reprises strictement à l’identique.

4) Taille tumorale et pièce opératoire
- La taille tumorale ne doit jamais être inventée ou estimée si elle n’est pas fournie.
- Si elle est fournie, elle doit être identique dans la macroscopie, la microscopie et la conclusion.
- La taille tumorale doit toujours être inférieure ou égale à la taille de la pièce opératoire.
- Si poids et dimensions de la pièce sont ajoutés, ils doivent être plausibles et cohérents entre eux.

5) Hormonorécepteurs et Ki67
- Si RE ou RP sont positifs, utiliser le pourcentage fourni dans les données cliniques.
- Si RE ou RP sont négatifs, les rapporter comme négatifs sans inventer d’intensité absente.
- Une seule valeur par marqueur, sans contradiction.

6) Ganglions lymphatiques
- Le nombre de ganglions atteints doit toujours être inférieur ou égal au nombre de ganglions examinés.
- Le nombre décrit en macroscopie, microscopie et conclusion doit être identique.
- Si des ganglions sont examinés et qu’il n’y a aucune métastase ganglionnaire, indiquer 0/N dans la conclusion (N = nombre de ganglions examinés).
- Si le statut ganglionnaire est non évalué, ne pas mentionner de nombre de ganglions ni de statut (pas de 0/0, pN0, etc.).
- Ne pas mentionner de rupture capsulaire si les ganglions sont négatifs.


7) Grade
- Le grade SBR ne s’applique qu’aux lésions infiltrantes.
- Si un score chiffré est présent (ex. 3 + 2 + 1), calculer le total et en déduire le grade :
  - 3 à 5 : grade I
  - 6 à 7 : grade II
  - 8 à 9 : grade III
- Corriger toute incohérence entre score et grade.

────────────────────────────────────────
RÈGLES DE STYLE
────────────────────────────────────────
- Français médical clair, structuré, homogène, sans redondance.
- Fournir uniquement le rapport final.
- N’ajouter aucune explication, justification ou note hors compte rendu.
"""

    if system_prompt:
        return [
            {
                "role": "system",
                "content": (
                    "Tu es un médecin anatomopathologiste hospitalier expérimenté. "
                    "Tu rédiges des comptes rendus anatomopathologiques complets, cohérents "
                    "et conformes aux standards hospitaliers français. "
                    "Tu corriges automatiquement toute incohérence médicale, stylistique ou logique."
                ),
            },
            {
                "role": "user",
                "content": user_prompt,
            },
        ]

    inline_system_prompt = """RÔLE :
Tu es un médecin anatomopathologiste hospitalier expérimenté.
Tu rédiges des comptes rendus anatomopathologiques complets, cohérents et conformes aux standards hospitaliers français.
Tu corriges automatiquement toute incohérence médicale, stylistique ou logique.

"""

    return [
        {
            "role": "user",
            "content": f"{inline_system_prompt}{user_prompt}",
        }
    ]

## Load tokenizer, base model, and fine-tuned LoRA adapter

In [6]:
GEMMA_ASSISTANT_ONLY_CHAT_TEMPLATE = """{{ bos_token }}
{%- if messages[0]['role'] == 'system' -%}
    {%- set first_user_prefix = messages[0]['content'] + '\\n\\n' -%}
    {%- set loop_messages = messages[1:] -%}
{%- else -%}
    {%- set first_user_prefix = "" -%}
    {%- set loop_messages = messages -%}
{%- endif -%}

{%- for message in loop_messages -%}
    {%- if (message['role'] == 'user') != (loop.index0 % 2 == 0) -%}
        {{ raise_exception('Conversation roles must alternate user/assistant/user/assistant/...') }}
    {%- endif -%}

    {%- if message['role'] == 'user' -%}
        {{ '<start_of_turn>user\\n' + first_user_prefix + (message['content'] | trim) + '<end_of_turn>\\n' }}
    {%- elif message['role'] == 'assistant' -%}
        {{ '<start_of_turn>model\\n' }}
        {%- generation -%}
            {{ (message['content'] | trim) + '<end_of_turn>\\n' }}
        {%- endgeneration -%}
    {%- else -%}
        {{ raise_exception('Only user and assistant roles are supported, except optional initial system message.') }}
    {%- endif -%}
{%- endfor -%}

{%- if add_generation_prompt -%}
    {{ '<start_of_turn>model\\n' }}
{%- endif -%}
"""

In [7]:
def load_medgemma_tokenizer(base_model_dir, lora_adapter_dir):
    try:
        tokenizer = AutoTokenizer.from_pretrained(
            lora_adapter_dir,
            trust_remote_code=True,
        )
        print(f"Loaded tokenizer from LoRA adapter dir: {lora_adapter_dir}")
    except Exception as e:
        print(f"Could not load tokenizer from LoRA adapter dir: {e}")
        print(f"Falling back to base tokenizer: {base_model_dir}")

        tokenizer = AutoTokenizer.from_pretrained(
            base_model_dir,
            trust_remote_code=True,
        )

        # Critical: restore the same template used during training.
        tokenizer.chat_template = GEMMA_ASSISTANT_ONLY_CHAT_TEMPLATE

    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    tokenizer.padding_side = "left"

    return tokenizer

In [8]:
def load_finetuned_model(
    base_model_dir=BASE_MODEL_DIR,
    lora_adapter_dir=LORA_ADAPTER_DIR,
    merge_lora=MERGE_LORA,
):
    tokenizer = load_medgemma_tokenizer(BASE_MODEL_DIR, LORA_ADAPTER_DIR)

    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "left"

    base_model = AutoModelForCausalLM.from_pretrained(
        base_model_dir,
        torch_dtype=torch.bfloat16,
        device_map="auto",
        cache_dir="./cache",
        trust_remote_code=True,
    )

    model = PeftModel.from_pretrained(
        base_model,
        lora_adapter_dir,
    )

    if merge_lora:
        print("Merging LoRA adapter into base model...")
        model = model.merge_and_unload()

    model.eval()
    return tokenizer, model


tokenizer, model = load_finetuned_model()
print("Loaded fine-tuned model.")

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loaded tokenizer from LoRA adapter dir: ../../FineTuneGeneration/models/Ddkd_1_2_medgemma-27b-report-generation-lora/


Loading weights:   0%|          | 0/808 [00:00<?, ?it/s]

Loaded fine-tuned model.


## Inference function

In [9]:
def clean_generated_text(text):
    text = text.strip()
    if "</think>" in text:
        text = text.split("</think>", 1)[1].strip()
    return text


@torch.inference_mode()
def inference_model_finetuned(
    data,
    tokenizer,
    model,
    batch_size=1,
    greedy=True,
    system_prompt=False,
    max_input_length=8192,
    max_new_tokens_greedy=1200,
    max_new_tokens_sampling=1300,
):
    data_to_save = []
    pbar = tqdm(total=len(data), desc="Generating completions")

    for i in range(0, len(data), batch_size):
        batch = data.iloc[i:i + batch_size]

        batch_messages = []
        entry_datas = []
        indexes = []
        donnees_cliniques_batch = []

        for idx, row in batch.iterrows():
            entry_data = row.to_dict()
            indexes.append(idx)
            entry_datas.append(entry_data)

            donnees_cliniques = create_donnees_cliniques(entry_data)
            donnees_cliniques_batch.append(donnees_cliniques)

            message = generate_report_prompt(donnees_cliniques, system_prompt=system_prompt)
            batch_messages.append(message)

        try:
            text_batch = tokenizer.apply_chat_template(
                batch_messages,
                tokenize=False,
                add_generation_prompt=True,
                enable_thinking=False,
            )
        except TypeError:
            text_batch = tokenizer.apply_chat_template(
                batch_messages,
                tokenize=False,
                add_generation_prompt=True,
            )

        model_inputs_batch = tokenizer(
            text_batch,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=max_input_length,
        ).to(model.device)

        # add stop token
        end_of_turn_id = tokenizer.convert_tokens_to_ids("<end_of_turn>")
        
        eos_token_ids = [tokenizer.eos_token_id]
        
        if end_of_turn_id is not None and end_of_turn_id != tokenizer.unk_token_id:
            eos_token_ids.append(end_of_turn_id)
        
        # print("eos_token:", tokenizer.eos_token, tokenizer.eos_token_id)
        # print("<end_of_turn>:", end_of_turn_id)
        # print("stop ids:", eos_token_ids)

        
        if greedy:
            generated_ids_batch = model.generate(
                **model_inputs_batch,
                max_new_tokens=max_new_tokens_greedy,
                do_sample=False,
                num_beams=1,
                repetition_penalty=1.1,
                length_penalty=1.0,
                early_stopping=True,
                pad_token_id=tokenizer.pad_token_id,
                eos_token_id=eos_token_ids,
            )
        else:
            generated_ids_batch = model.generate(
                **model_inputs_batch,
                max_new_tokens=max_new_tokens_sampling,
                # min_new_tokens=200,
                do_sample=True,
                temperature=0.2,
                top_p=0.9,
                top_k=0,
                repetition_penalty=1.1,
                length_penalty=1.0,
                pad_token_id=tokenizer.pad_token_id,
                eos_token_id=eos_token_ids,
            )

        # With left padding, slicing by the padded input length is correct for the whole batch.
        generated_ids_batch = generated_ids_batch[:, model_inputs_batch.input_ids.shape[1]:]
        generated_text_batch = tokenizer.batch_decode(generated_ids_batch, skip_special_tokens=True)
        generated_text_batch = [clean_generated_text(t) for t in generated_text_batch]

        for idx, entry_data, donnees_cliniques, prompt_messages, generated_text in zip(
            indexes,
            entry_datas,
            donnees_cliniques_batch,
            batch_messages,
            generated_text_batch,
        ):
            data_to_save.append({
                "index": int(idx) if isinstance(idx, (int, np.integer)) else idx,
                "entry_data": entry_data,
                "donnees_cliniques": donnees_cliniques,
                "prompt_messages": prompt_messages,
                "generated_text": generated_text,
            })

        pbar.update(len(batch))
        pbar.set_postfix({"batch": f"{min(i + batch_size, len(data))}/{len(data)}"})

    pbar.close()
    return data_to_save

## Load CSV and preview one prompt

In [10]:
df_data_patient = pd.read_csv(INPUT_FILE)
print("Rows:", len(df_data_patient))
print("Columns:", list(df_data_patient.columns))

df_data_patient.head()

Rows: 1061
Columns: ['type_diagnostique', 'ganglions_preleves', 'ganglions_atteints', 'rupture_capsulaire', 'taille_tumor_0', 're_tumor_0', 'rp_tumor_0', 'embols_vasculaires_tumor_0', 'cerb_tumor_0', 'marges_saines_tumor_0', 'ki67_tumor_0', 'ref_morpho_name_tumor_0', 'ref_grade_tumor_0', 're_perc', 'rp_perc']


,type_diagnostique,ganglions_preleves,ganglions_atteints,rupture_capsulaire,taille_tumor_0,re_tumor_0,rp_tumor_0,embols_vasculaires_tumor_0,cerb_tumor_0,marges_saines_tumor_0,ki67_tumor_0,ref_morpho_name_tumor_0,ref_grade_tumor_0,re_perc,rp_perc
0,biopsie,unknown,unknown,unknown,unknown,négatifs,positifs,unknown,négatif,unknown,unknown,adenocarcinome canalaire infiltrant,SBR2,0,90
1,biopsie,unknown,unknown,unknown,unknown,négatifs,positifs,unknown,négatif,unknown,unknown,adenocarcinome canalaire infiltrant,SBR2,0,20
2,biopsie,unknown,unknown,unknown,unknown,négatifs,positifs,unknown,négatif,unknown,unknown,adenocarcinome canalaire infiltrant,SBR2,0,50
3,mammectomie,2,0,unknown,12,négatifs,positifs,0,négatif,unknown,8,adenocarcinome canalaire infiltrant,unknown,0,100
4,tumorectomie,3,0,unknown,78,positifs,négatifs,unknown,négatif,oui,unknown,carcinome lobulaire,SBR3,90,0


In [11]:
example_entry = df_data_patient.iloc[0].to_dict()
example_donnees = create_donnees_cliniques(example_entry)
example_messages = generate_report_prompt(example_donnees, system_prompt=SYSTEM_PROMPT)

print("========== DONNÉES CLINIQUES ==========")
print(example_donnees)

print("========== RENDERED PROMPT PREFIX ==========")
try:
    rendered = tokenizer.apply_chat_template(
        example_messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False,
    )
except TypeError:
    rendered = tokenizer.apply_chat_template(
        example_messages,
        tokenize=False,
        add_generation_prompt=True,
    )

print(rendered[:3000])

========== DONNÉES CLINIQUES ==========
- Échantillon : biopsie
- Diagnostic (morphologie) : adenocarcinome canalaire infiltrant
- Grade SBR : SBR2
- Taille tumorale : non applicable (biopsie)
- Nombre de ganglions examinés : non évalués
- Emboles vasculaires : non évalués
- Présence de rupture capsulaire : non évaluée
- RE ou RO (récepteurs aux œstrogènes) : négatifs
- RP (récepteurs à la progestérone) : positifs ; pourcentage : 90%
- HER2 (CerbB2) : 1+
- Prolifération cellulaire (Ki67) : non évaluée
- Marges chirurgicales saines : non applicable (biopsie)

========== RENDERED PROMPT PREFIX ==========
<bos><start_of_turn>user
RÔLE :
Tu es un médecin anatomopathologiste hospitalier expérimenté.
Tu rédiges des comptes rendus anatomopathologiques complets, cohérents et conformes aux standards hospitaliers français.
Tu corriges automatiquement toute incohérence médicale, stylistique ou logique.

Tache:
Rédige un **compte rendu anatomopathologique complet en français médical** à partir des

## Generate one example first

In [12]:
one_result = inference_model_finetuned(
    df_data_patient.iloc[:1],
    tokenizer=tokenizer,
    model=model,
    batch_size=1,
    greedy=GREEDY,
    system_prompt=SYSTEM_PROMPT,
    max_input_length=MAX_INPUT_LENGTH,
    max_new_tokens_greedy=MAX_NEW_TOKENS_GREEDY,
    max_new_tokens_sampling=MAX_NEW_TOKENS_SAMPLING,
)

print(one_result[0]["generated_text"])

Generating completions:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] The following generation flags are not valid and may be ignored: ['early_stopping']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


## COMPTE RENDU ANATOMOPATHOLOGIQUE

**Titre :** MICROBIOPSIES MAMMAIRES : ADÉNOCARCINOME CANALAIRE INFILTRANT

**Examen macroscopique :**
Fragments de biopsies mammaires mesurant ensemble 1,4 cm dans leur plus grande dimension. Les prélèvements sont entièrement inclus.

**Examen microscopique :**
On observe une prolifération néoplasique maligne mammaire primitive faite de massifs et de travées infiltrants. Les atypies sont modérées. Les mitoses sont peu nombreuses. Le stroma est fibreux desmoplastique.

**Étude immunohistochimique :**
*   Récepteurs hormonaux :
    *   Œstrogènes (RE) : négatifs.
    *   Progestérone (RP) : positifs ; 90 % de cellules exprimées.
*   HER2 (CerbB2) : 0.
*   Prolifération cellulaire (Ki67) : non évaluée.

**Conclusion :**
Adénocarcinome mammaire infiltrant de type canalaire évalué sur biopsies à un grade SBR 2.
Statut des récepteurs hormonaux : RE négatif, RP positif (90%).
Statut Her2Neu : 0.
Indice de prolifération Ki67 : non évalué.
Nombre de ganglion

## Generate the full file

In [1]:
BATCH_SIZE=8
results = inference_model_finetuned(
    df_data_patient,
    tokenizer=tokenizer,
    model=model,
    batch_size=BATCH_SIZE,
    greedy=GREEDY,
    system_prompt=SYSTEM_PROMPT,
    max_input_length=MAX_INPUT_LENGTH,
    max_new_tokens_greedy=MAX_NEW_TOKENS_GREEDY,
    max_new_tokens_sampling=MAX_NEW_TOKENS_SAMPLING,
)

print("Generated reports:", len(results))

## Save results

In [ ]:
results[:1]

In [ ]:
greedy_text = "greedy" if GREEDY else "no_greedy"
output_json = os.path.join(OUTPUT_FOLDER, f"output1-2_{greedy_text}_medgemma27b_lora.json")
output_jsonl = os.path.join(OUTPUT_FOLDER, f"output1-2_{greedy_text}_medgemma27b_lora.jsonl")
output_csv = os.path.join(OUTPUT_FOLDER, f"output1-2_{greedy_text}_medgemma27b_lora.csv")

with open(output_json, "w", encoding="utf-8") as f:
    json.dump(results, f, ensure_ascii=False, indent=2)

with open(output_jsonl, "w", encoding="utf-8") as f:
    for row in results:
        f.write(json.dumps(row, ensure_ascii=False) + "\n")


flat_rows = []
for row in results:
    flat = {
        "index": row["index"],
        "donnees_cliniques": row["donnees_cliniques"],
        "generated_text": row["generated_text"],
    }
    # Keep original entry columns too, prefixed to avoid collisions.
    for k, v in row["entry_data"].items():
        flat[f"entry_{k}"] = v
    flat_rows.append(flat)

pd.DataFrame(flat_rows).to_csv(output_csv, index=False)

print("Saved JSON:", output_json)
print("Saved JSONL:", output_jsonl)
print("Saved CSV:", output_csv)

## Inspect generated reports

In [ ]:
for row in results[:3]:
    print("=" * 100)
    print("INDEX:", row["index"])
    print(row["generated_text"])